In [50]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

PATH = "oofs/"  # folder where you put all the CSV pairs
true = pd.read_csv('dataset/training_data.csv', header=None,
                   delimiter="\t", names=["labels", "text"])["labels"].values

files = []
x_train = []
x_test = []

for fname in os.listdir(PATH):
    if fname.endswith('.csv') and '_oof' not in fname: # Not _oof file 
        base = fname[:-4] # Get both csv files
        oof_path = os.path.join(PATH, base + '_oof.csv')
        pred_path = os.path.join(PATH, fname)
        if not os.path.exists(oof_path):
            continue
        print(f"Loading {base}")
        x_train.append(pd.read_csv(oof_path).values.flatten())
        x_test.append(pd.read_csv(pred_path).values.flatten())
        files.append(base)

x_train = np.stack(x_train).T  # shape: (n_train, n_models)
x_test = np.stack(x_test).T    # shape: (n_test, n_models)

# Meta-learner
meta = LogisticRegression()
meta.fit(x_train, true)

final_probs = meta.predict_proba(x_test)[:, 1]
final_preds = (final_probs >= 0.5).astype(int)

oof_preds = (meta.predict_proba(x_train)[:, 1] >= 0.5).astype(int)
print(f"Ensemble OOF accuracy: {accuracy_score(true, oof_preds):.4f}")
print(f"Ensemble OOF F1: {f1_score(true, oof_preds):.4f}")

Loading MLP_medium_mpnet
Loading LogisticRegression_TFIDF
Loading LinearSVC_TFIDF_bigram
Loading LR_mpnet
Loading LR_MiniLM_C10
Loading LR_MiniLM_C0.1
Loading LinearSVC_mpnet
Loading LR_TFIDF_bigram
Loading MultinomialNB_BOW
Loading RandomForestClassifier_TFIDF
Loading LR_TFIDF_100k
Loading LinearSVC_TFIDF_100k
Loading LSTM_small
Loading MLP_small_MiniLM
Loading MLP_medium_MiniLM
Loading MultinomialNB
Loading RandomForestClassifier
Loading LSTM_deep
Loading LR_TFIDF_char
Loading LR_MiniLM_C0.01
Loading MultinomialNB_TFIDF
Loading MLP_small_mpnet
Loading Ridge_MiniLM
Loading LinearSVC_TFIDF_char
Loading GradientBoostingClassifier
Loading SGD_log_MiniLM
Loading DistilBERT_2
Loading NB_TFIDF_char
Loading RandomForestClassifier_BOW
Loading LinearSVC_MiniLM
Loading DistilBERT_1
Loading DistilBERT_3ep_1
Loading StackingClassifier
Loading LogisticRegression_BOW
Loading LogisticRegression
Ensemble OOF accuracy: 0.9850
Ensemble OOF F1: 0.9845


In [49]:
test_data = pd.read_csv('dataset/testing_data.csv', header=None, delimiter="\t", names=["label","text"])
test_data["label"] = final_preds
test_data.to_csv("final_submission.csv", index=False, sep="\t", header=False)